[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/adan-rs/amd/blob/main/extras/03_Canasta_de_compra.ipynb)
# Canasta de compra (Análisis de afinidad)

*¿Para qué se utiliza?*
El análisis de afinidad busca identificar qué elementos tienden a aparecer juntos con frecuencia dentro de un conjunto de transacciones o eventos. Su aplicación más conocida es el "análisis de canasta" (*basket analysis*): identificar qué artículos son comprados al mismo tiempo. También se utiliza en detección de fraudes (transacciones con combinaciones inusuales), segmentación de mercados y sistemas de recomendación.

Ejemplos de uso en negocios:
- Una cadena de supermercados analiza los tickets de compra para decidir qué productos colocar cerca uno del otro, o qué combos ofrecer en promoción.
- Una plataforma de streaming utiliza los patrones de consumo de sus usuarios para recomendar "contenido que también podría interesarte".
- Un banco revisa combinaciones inusuales de transacciones en una cuenta (por ejemplo, un patrón de compras que casi nunca ocurre junto) como señal de posible fraude.

*Variables consideradas*
Un conjunto de transacciones, donde cada transacción (compra, sesión, usuario) está asociada a uno o más ítems (productos, películas, páginas visitadas). Los datos se organizan como una matriz binaria transacción × ítem, donde cada celda indica si ese ítem estuvo presente (1) o no (0) en esa transacción: no importa la cantidad, solo la presencia.

*¿Cómo funciona?*
El algoritmo que utilizaremos será el *algoritmo a priori*, que incluye los siguientes pasos:
1. Cálculo de la frecuencia de los ítems. Se identifican los elementos individuales más frecuentes y se descartan aquellos cuyo *soporte* (proporción de transacciones en que aparece) sea menor a cierto umbral.
2. Generación de conjuntos de ítems frecuentes. Se combinan los ítems frecuentes en pares, tríos, etc., para formar conjuntos más grandes. Se descartan los conjuntos cuyo soporte sea menor al umbral.
3. Extracción de reglas de asociación. Se generan reglas del tipo "si compras X, es probable que compres Y".

*Supuestos y recomendaciones*
- Se requiere un número razonable de transacciones: con pocas, cualquier coincidencia parece un patrón fuerte.
- El umbral mínimo de soporte es una decisión del analista: un umbral muy bajo genera un número enorme de reglas, muchas de ellas espurias o poco útiles; un umbral muy alto puede descartar combinaciones poco frecuentes pero interesantes desde el punto de vista de negocio.
- Los datos deben poder representarse en formato binario (presencia/ausencia); si interesa la cantidad comprada de cada ítem, se requieren variantes del algoritmo que quedan fuera del alcance de este notebook.

## Ejemplo: Recomendación de películas

El archivo `peliculas.csv` contiene información de las evaluaciones de películas realizadas por los usuarios de un sitio web. Cada usuario está representado por un id y no se proporciona ninguna otra información personal. Los datos provienen originalmente de la base "MovieLens Beliefs Dataset 2024" en el sitio http://grouplens.org/datasets/movielens y los datos fueron filtrados para mostrar solo películas del año 2020 y posteriores, con más de 50 evaluaciones. Las calificaciones de cada película va desde 0.5 estrellas hasta 5 estrellas.

In [ ]:
import pandas as pd
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

import warnings
warnings.simplefilter(action='ignore', category=DeprecationWarning)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
df = pd.read_excel('https://github.com/adan-rs/amd/raw/main/data/movies2.xlsx')

In [ ]:
# Convertir ratings > 3 en "me gusta"
likes_matrix = df.copy()
likes_matrix['liked'] = (likes_matrix['rating'] > 3).astype(int)
likes_matrix.sample(5)

In [ ]:
# Crear matriz usuario-película
user_movie_matrix = likes_matrix.pivot_table(index='userId',columns='title',
    values='liked', fill_value=0)

# Verificar que todos los valores son binarios (0 o 1)
user_movie_matrix = user_movie_matrix.astype(bool).astype(int)
user_movie_matrix

In [ ]:
# Encontrar conjuntos frecuentes
frequent_itemsets = apriori(user_movie_matrix, 
                          min_support=0.1,  # Ajustar soporte
                          use_colnames=True)

In [ ]:
frequent_itemsets

In [ ]:
# Generar reglas de asociación
rules = association_rules(frequent_itemsets, 
                        metric="confidence",
                        min_threshold=0.5)  # Ajusta este valor según tus necesidades

*Criterio de evaluación del ajuste*

A diferencia de una prueba de hipótesis, el análisis de afinidad no tiene un criterio de rechazo/no rechazo: cada regla generada se evalúa con tres métricas:
- **Support (soporte)**: proporción de transacciones en las que aparecen juntos el antecedente y el consecuente. Indica qué tan frecuente es el patrón.
- **Confidence (confianza)**: probabilidad condicional de que ocurra el consecuente dado que ya ocurrió el antecedente. Indica qué tan confiable es la regla.
- **Lift**: cuánto más probable es el consecuente cuando ocurre el antecedente, en comparación con su probabilidad base. Un lift mayor a 1 indica una asociación positiva (más fuerte cuanto más se aleje de 1); un lift cercano a 1 indica que no hay relación real más allá del azar.

Una regla con confianza alta pero soporte muy bajo debe interpretarse con cautela: puede reflejar una coincidencia poco representativa en lugar de un patrón útil para el negocio.

In [ ]:
# Ordenar reglas por lift
rules = rules.sort_values('lift', ascending=False)
rules.head(5)

In [ ]:
def recomendar_items(item, rules_df, n_recomendaciones=5):
    """
    Obtiene recomendaciones únicas para un item específico basado en reglas de asociación
    """
    # Filtrar reglas donde el item dado está en los antecedentes
    item_rules = rules_df[rules_df['antecedents'].apply(lambda x: item in str(x))].copy()
    
    if len(item_rules) == 0:
        return "No se encontraron recomendaciones para este item"
    
    # Crear nueva columna con items individuales usando loc
    item_rules.loc[:, 'item'] = item_rules['consequents'].apply(lambda x: list(x)[0])
    
    # Quedarnos con la mejor regla (mayor lift) para cada item
    best_rules = item_rules.sort_values('lift', ascending=False)\
                          .drop_duplicates(subset=['item'], keep='first')
    
    # Seleccionar las columnas relevantes y renombrarlas para mayor claridad
    recommendations = best_rules[['item', 'confidence', 'lift']].copy()
    recommendations = recommendations.head(n_recomendaciones)
    recommendations = recommendations.rename(columns={'item': 'item_recomendado'})
    
    # Formatear los valores numéricos
    recommendations.loc[:, 'confidence'] = recommendations['confidence'].apply(lambda x: f"{x:.2%}")
    recommendations.loc[:, 'lift'] = recommendations['lift'].apply(lambda x: f"{x:.2f}")
    
    return recommendations

In [ ]:
pelicula_ejemplo = 'The Batman (2022)'
recomendaciones = recomendar_items(pelicula_ejemplo, rules)
print(f"Recomendaciones para {pelicula_ejemplo}:")
print(recomendaciones)

**Ejemplo de reporte de resultados**:
>"Se aplicó un análisis de afinidad (algoritmo a priori) sobre las calificaciones de usuarios a películas recientes, considerando como 'me gusta' las calificaciones mayores a 3 estrellas. Con un soporte mínimo de 0.10 y una confianza mínima de 0.50, se identificaron 102 conjuntos frecuentes de películas. Para la película 'The Batman (2022)', las recomendaciones con mayor lift fueron 'Spider-Man: No Way Home' (confianza = 58.2%, lift = 2.01), 'Dune' (confianza = 75.8%, lift = 1.83) y 'Everything Everywhere All at Once' (confianza = 69.0%, lift = 1.65). Un lift superior a 1 en los tres casos indica que a los usuarios que vieron 'The Batman' les resultó considerablemente más probable haber visto estas películas que al usuario promedio, lo que las convierte en candidatas razonables para un sistema de recomendación."

## Ejercicio

1. Repite el análisis utilizando un umbral de soporte mínimo más bajo (por ejemplo, `min_support=0.05`). ¿Cuántos conjuntos frecuentes adicionales aparecen? ¿Todas las nuevas reglas te parecen igual de confiables que las originales?
2. Elige otra película del catálogo (`user_movie_matrix.columns`) y utiliza `recomendar_items()` para obtener sus recomendaciones. Con base en el soporte, la confianza y el lift, ¿recomendarías usar estas reglas en un sistema real de recomendación?

## Referencias
- Documentación de `mlxtend`: http://rasbt.github.io/mlxtend/user_guide/frequent_patterns/apriori/
- Datos originales: MovieLens Beliefs Dataset 2024, http://grouplens.org/datasets/movielens